# CS3244 Group 17 Project: IMDb Spoiler Detection from Movie Reviews

This notebook presents an end-to-end workflow for binary spoiler classification using IMDb review text.

Workflow: setup and data loading, exploratory data analysis (EDA), data cleaning and preprocessing, data splitting, feature engineering, model training and model evaluation.

## 0. Setup and Data Loading

In [2]:
# imports
import pandas as pd
import numpy as np
import joblib, re, html, unicodedata
from pathlib import Path
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, fbeta_score, precision_score, recall_score, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

In [3]:
# Read review dataset
df = pd.read_json("IMDB_reviews.json", lines=True)

In [173]:
# Read movies dataset
df_movies = pd.read_json("IMDB_movie_details.json", lines=True)

## 1. Exploratory Data Analysis

### 1.1 EDA of reviews dataset

In [5]:
# Check details of reviews dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573913 entries, 0 to 573912
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   review_date     573913 non-null  object
 1   movie_id        573913 non-null  object
 2   user_id         573913 non-null  object
 3   is_spoiler      573913 non-null  bool  
 4   review_text     573913 non-null  object
 5   rating          573913 non-null  int64 
 6   review_summary  573913 non-null  object
dtypes: bool(1), int64(1), object(5)
memory usage: 26.8+ MB


In [6]:
print(df.isnull().sum())
# No sign of missing entries initially, but requires further checks into empty strings

review_date       0
movie_id          0
user_id           0
is_spoiler        0
review_text       0
rating            0
review_summary    0
dtype: int64


In [7]:
print(df.shape)
# 573,913 reviews, 7 features

(573913, 7)


In [8]:
print(df.columns)

Index(['review_date', 'movie_id', 'user_id', 'is_spoiler', 'review_text',
       'rating', 'review_summary'],
      dtype='object')


In [9]:
print(df["review_text"].str.len().describe())
# Mean character length is 1460, relatively large inputs

count    573913.000000
mean       1460.553525
std        1125.577019
min          18.000000
25%         719.000000
50%        1052.000000
75%        1815.000000
max       14963.000000
Name: review_text, dtype: float64


In [10]:
print(df["is_spoiler"].value_counts(normalize=True))
# Dataset is slightly imbalanced (73.7% non-spoiler vs 26.3% spoiler)

is_spoiler
False    0.737026
True     0.262974
Name: proportion, dtype: float64


In [11]:
print((df["review_text"].str.strip() == "").sum())
# No empty or whitespace-only reviews detected, so no need for removal of trivial empty entries

0


In [12]:
print(df.duplicated(subset=["review_text"]).sum())
# 528 duplicate review texts, redundant samples that may cause bias or data leakage if not removed

528


In [13]:
dup_text = df[df.duplicated(subset=["review_text"], keep=False)]
dup_text.groupby("review_text").size().sort_values(ascending=False).head(10)
# Some duplicated text are quite long, indicating repeated reviews, should remove

review_text
I have never seen such an amazing film since I saw The Shawshank Redemption. Shawshank encompasses friendships, hardships, hopes, and dreams. And what is so great about the movie is that it moves you, it gives you hope. Even though the circumstances between the characters and the viewers are quite different, you don't feel that far removed from what the characters are going through.It is a simple film, yet it has an everlasting message. Frank Darabont didn't need to put any kind of outlandish special effects to get us to love this film, the narration and the acting does that for him. Why this movie didn't win all seven Oscars is beyond me, but don't let that sway you to not see this film, let its ranking on the IMDb's top 250 list sway you, let your friends recommendation about the movie sway you.Set aside a little over two hours tonight and rent this movie. You will finally understand what everyone is talking about and you will understand why this is my all time favorite m

In [14]:
print(df.groupby("review_text")["is_spoiler"].nunique().value_counts())
# Almost all duplicate reviews have consistent labels (573,307), but 78 cases show conflicting labels, minor label noise

is_spoiler
1    573307
2        78
Name: count, dtype: int64


In [15]:
df["length"] = df["review_text"].str.len()
print(df.groupby("is_spoiler")["length"].describe())
# Spoiler reviews are significantly longer on average (~1888 vs ~1308 chars), suggesting length is a strong predictive signal

               count         mean          std   min    25%     50%     75%  \
is_spoiler                                                                    
False       422989.0  1308.154425  1021.050440  18.0  682.0   947.0  1575.0   
True        150924.0  1887.676731  1283.848098  50.0  920.0  1459.0  2456.0   

                max  
is_spoiler           
False       14963.0  
True        14302.0  


In [16]:
print(df.groupby("is_spoiler")["rating"].mean())
# Spoiler reviews tend to have lower average ratings (~6.52 vs ~7.11), indicating a potential correlation between sentiment and spoiler likelihood

is_spoiler
False    7.110031
True     6.517665
Name: rating, dtype: float64


In [17]:
print(df["movie_id"].value_counts().describe())
# Review distribution is highly uneven across movies (max ~4845 reviews), indicating potential movie-level bias in the dataset

count    1572.000000
mean      365.084606
std       283.088117
min         2.000000
25%       165.000000
50%       326.000000
75%       529.000000
max      4845.000000
Name: count, dtype: float64


In [18]:
print(df["user_id"].value_counts().describe())
# Most users contribute only one review, but a few users are highly active (up to 1303 reviews), introducing potential user-level bias

count    263407.000000
mean          2.178807
std          10.665784
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        1303.000000
Name: count, dtype: float64


In [19]:
print((df["review_text"] == df["review_summary"]).sum())
# Review text and summary are almost always different (only 1 identical case), indicating they provide complementary information

1


In [18]:
print(df[df["review_text"].str.len() > 10000].shape)
# Only 25 reviews exceed 10,000 characters, indicating rare but extreme outliers that may impact model efficiency

(25, 8)


In [19]:
print(df["review_text"].str.contains("dies|ending|killer", case=False).groupby(df["is_spoiler"]).mean())
# Spoiler reviews are more likely to contain keywords like "dies", "ending", or "killer" (~27% vs ~17%), indicating partial keyword-driven signal.

is_spoiler
False    0.168489
True     0.271534
Name: review_text, dtype: float64


In [20]:
# Review date stored as object, convert to datetime
df["review_date"] = pd.to_datetime(df["review_date"])

### 1.2 EDA of movies dataset

In [175]:
# Check details of movies dataset
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1572 entries, 0 to 1571
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   movie_id       1572 non-null   object 
 1   plot_summary   1572 non-null   object 
 2   duration       1572 non-null   object 
 3   genre          1572 non-null   object 
 4   rating         1572 non-null   float64
 5   release_date   1572 non-null   object 
 6   plot_synopsis  1572 non-null   object 
dtypes: float64(1), object(6)
memory usage: 86.1+ KB


In [176]:
# Check that all entries have unique movie_ids
df_movies["movie_id"].nunique(), len(df_movies)

(1572, 1572)

In [177]:
# Check for missing / unsused movies
review_movie_ids = set(df["movie_id"])
movie_ids = set(df_movies["movie_id"])

# Reviews without metadata
missing_movies = review_movie_ids - movie_ids

# Metadata without reviews
unused_movies = movie_ids - review_movie_ids

print("Missing metadata:", len(missing_movies))
print("Unused metadata:", len(unused_movies))

Missing metadata: 2
Unused metadata: 2


In [178]:
# There were 2 missing and 2 unused, looking further into the discrepency
print("Missing:", missing_movies)
print("Unused:", unused_movies)

# Missing: {'tt0104014', 'tt0114142'}
# Unused: {'tt0114142/', 'tt0104014/'}
# Inconsistency was due to '/'

Missing: {'tt0114142', 'tt0104014'}
Unused: {'tt0114142/', 'tt0104014/'}


In [179]:
# Find all movie_ids ending with '/'
df_movies[df_movies["movie_id"].str.endswith("/")]
# Its those 2 movies only: isolated issue, solve later in data cleaning

,movie_id,plot_summary,duration,genre,rating,release_date,plot_synopsis
1570,tt0104014/,"For a while now, beautiful 24-year-old Diana B...",1h 33min,"[Comedy, Drama]",5.3,1992-02-21,
1571,tt0114142/,"The marriage of David Burgess, a senior execut...",1h 32min,"[Drama, Thriller]",4.0,1999-01-29,


In [180]:
# Checking details of plot summaries
df_movies["plot_summary"].str.len().describe()

count    1572.000000
mean      614.258270
std       240.194629
min        95.000000
25%       423.750000
50%       578.000000
75%       783.500000
max      1077.000000
Name: plot_summary, dtype: float64

In [181]:
# Checking details of plot synopses
df_movies["plot_synopsis"].str.len().describe()
# At least one movie synopsis has length of zero

count     1572.000000
mean      8214.993639
std       8435.385266
min          0.000000
25%       2898.750000
50%       6192.500000
75%      10871.000000
max      63904.000000
Name: plot_synopsis, dtype: float64

In [182]:
# Check for movie(s) with synopsis of length 0
df_movies[df_movies["plot_synopsis"].str.len() == 0]
# 233 movies with an empty plot synopsis

,movie_id,plot_summary,duration,genre,rating,release_date,plot_synopsis
2,tt0243655,"The setting is Camp Firewood, the year 1981. I...",1h 37min,"[Comedy, Romance]",6.7,2002-04-11,
10,tt0107131,"Three pets (Chance, a young dog unfamiliar wit...",1h 24min,"[Adventure, Comedy, Drama]",6.9,1993-02-12,
11,tt0110364,"In Urbania, Ohio, snobby ex-football star Kevi...",1h 47min,"[Comedy, Family, Sport]",6.3,1994-10-14,
22,tt0101700,The story is centered on a microcosm of a post...,1h 39min,"[Comedy, Crime]",7.7,1992-04-03,
24,tt0102768,Henry is a lawyer who survives a shooting only...,1h 48min,"[Drama, Romance]",6.7,1991-07-10,
...,...,...,...,...,...,...,...
1556,tt0285531,"Four childhood friends, Jonesy, Beaver, Pete a...",2h 14min,"[Drama, Horror, Sci-Fi]",5.5,2003-03-21,
1564,tt0914798,Young Bruno lives a wealthy lifestyle in prewa...,1h 34min,"[Drama, War]",7.8,2008-11-26,
1566,tt4047038,Centers on the titular holistic detective who ...,1h,"[Comedy, Mystery, Sci-Fi]",8.4,2017-04-01,
1570,tt0104014/,"For a while now, beautiful 24-year-old Diana B...",1h 33min,"[Comedy, Drama]",5.3,1992-02-21,


In [183]:
# Check genre type consistency
df_movies["genre"].apply(type).value_counts()

# flatten and count
genre_counts = df_movies["genre"].explode().value_counts()
print(genre_counts)

genre
Drama        799
Comedy       525
Action       438
Adventure    433
Crime        302
Romance      237
Thriller     233
Sci-Fi       188
Fantasy      178
Mystery      161
Family       155
Horror       116
Biography    101
Animation     84
History       58
Music         34
Sport         30
War           25
Musical       12
Western       10
Film-Noir      6
Name: count, dtype: int64


In [184]:
# Checking for genre structure edge cases
print(df_movies[df_movies["genre"].apply(lambda x: not isinstance(x, list))])
print(df_movies[df_movies["genre"].apply(len) == 0])

Empty DataFrame
Columns: [movie_id, plot_summary, duration, genre, rating, release_date, plot_synopsis]
Index: []
Empty DataFrame
Columns: [movie_id, plot_summary, duration, genre, rating, release_date, plot_synopsis]
Index: []


In [185]:
# Checking details of ratings
df_movies["rating"].astype(float).describe()

count    1572.000000
mean        7.071819
std         0.967966
min         2.400000
25%         6.500000
50%         7.100000
75%         7.800000
max         9.500000
Name: rating, dtype: float64

## 2. Data Cleaning and Preprocessing
This section performs cleaning and preprocessing to build the master dataset used across all downstream modeling steps.

### 2.1 Data Cleaning for reviews

In [32]:
# create master dataset
df_master = df.copy()

In [33]:
# Remove duplicate reviews which have different label assigned to the same text
conflicts = df_master.groupby("review_text")["is_spoiler"].nunique()
conflicts = conflicts[conflicts > 1].index

df_master = df_master[~df_master["review_text"].isin(conflicts)]

# Remove exact duplicate reviews to avoid redundancy and data leakage
df_master = df_master.drop_duplicates(subset=["review_text"])

# Remove extremely short reviews that are likely low-information or noisy
df_master = df_master[df_master["review_text"].str.len() > 25]

# Remove extremely long reviews that may distort feature distributions and increase computational cost
df_master = df_master[df_master["review_text"].str.len() < 10000]

# Add review length as a feature due to strong correlation with spoiler labels
df_master["length"] = df_master["review_text"].str.len()

# Reset index after row removals to keep the cleaned dataset tidy and consistent
df_master = df_master.reset_index(drop=True)

In [34]:
print(df_master.shape)
# Removed 634 reviews
# Additional feature is review length "length"

(573279, 8)


In [35]:
# Re-check class distribution after cleaning to ensure preprocessing did not heavily distort label balance
print(df_master["is_spoiler"].value_counts(normalize=True))
# Original: (73.7% non-spoiler vs 26.3% spoiler), class distribution remained very similar

is_spoiler
False    0.737009
True     0.262991
Name: proportion, dtype: float64


In [36]:
# Verify that exact duplicate reviews have been fully removed
print(df_master.duplicated(subset=["review_text"]).sum())
# No duplicates left

0


In [37]:
# Verify that no identical review text remains with conflicting spoiler labels
print(df_master.groupby("review_text")["is_spoiler"].nunique().value_counts())
# All review_texts have only 1 label

is_spoiler
1    573279
Name: count, dtype: int64


In [73]:
# Check that there are no invalid movie_ids
invalid_ids_reviews = df_master[~df_master["movie_id"].str.match(r"^tt\d+$", na=False)]
print(len(invalid_ids_reviews))

0


In [64]:
# Basic data cleaning function
def basic_clean(text):
    text = str(text) # Forces text to be a String
    text = html.unescape(text) # Converts HTML to normal text
    text = unicodedata.normalize("NFKC", text) # Normalizes unicode into a consistent canonical form
    text = re.sub(r"<[^>]+>", " ", text) # Removes HTML tags 
    text = text.lower() # Converts all text to lowercase
    text = re.sub(r"\s+", " ", text) # Collapses all whitespace into a single space
    text = text.strip() # Removes whitespace in front and behind of text
    return text

In [39]:
# Clean "clean_text" in master dataframe
df_master["clean_text"] = df_master["review_text"].apply(basic_clean)

In [29]:
# Compare clean_text to review_text
pd.set_option("display.max_colwidth", None)
df_master[["review_text", "clean_text"]].sample(3, random_state=42)

,review_text,clean_text
450069,"I'm sorry. I wanted to like it so much.I've been a fan of the show for years. I've watched every episode 10 times. I was so incredibly excited when I heard a movie was being made. Unfortunately, it wasn't the movie I was hoping to see.It's great to hear Carrie's voice again, talking about her and her friends little dramas. But even SATC got 'carried away' with that Hollywood cheesiness. Where the show was spicy and provocative, the movie was too long and moralizing. Though some of the dialogs where pretty good, a lot of others weren't. The half of the time I felt like I was watching 4 totally different people than I saw and the show 4 years ago. Also, the story was pitiful (Carrie marries Big - oh no, she doesn't! - wait, she does!), they just tried to squeeze every possible scenario in 144 minutes (alot of breaking up and getting back together). And even my two favorite gay men had like two lines in the whole movie! The only great thing in the movie was the wardrobe. Compliment to Patricia Field, she has done it again! (Though the designer logos were shooting of the screen). But when the critics said this movie was made 'just for the fans', they were wrong. This movie was made for people who have no clue of what Sex And The City is, and just want a fun girls night out. The fans however, will be disappointed.I'm sorry. I wanted to like it so much (I rate it '10' just on my personal account, so it wouldn't be suck on a 3,5, cause it doesn't deserve that either), but Sex and the City is just a mess of very very good fashion, and bad fart jokes. But I will buy it on DVD. Just to finish my collection.","i'm sorry. i wanted to like it so much.i've been a fan of the show for years. i've watched every episode 10 times. i was so incredibly excited when i heard a movie was being made. unfortunately, it wasn't the movie i was hoping to see.it's great to hear carrie's voice again, talking about her and her friends little dramas. but even satc got 'carried away' with that hollywood cheesiness. where the show was spicy and provocative, the movie was too long and moralizing. though some of the dialogs where pretty good, a lot of others weren't. the half of the time i felt like i was watching 4 totally different people than i saw and the show 4 years ago. also, the story was pitiful (carrie marries big - oh no, she doesn't! - wait, she does!), they just tried to squeeze every possible scenario in 144 minutes (alot of breaking up and getting back together). and even my two favorite gay men had like two lines in the whole movie! the only great thing in the movie was the wardrobe. compliment to patricia field, she has done it again! (though the designer logos were shooting of the screen). but when the critics said this movie was made 'just for the fans', they were wrong. this movie was made for people who have no clue of what sex and the city is, and just want a fun girls night out. the fans however, will be disappointed.i'm sorry. i wanted to like it so much (i rate it '10' just on my personal account, so it wouldn't be suck on a 3,5, cause it doesn't deserve that either), but sex and the city is just a mess of very very good fashion, and bad fart jokes. but i will buy it on dvd. just to finish my collection."
495181,"20 years after this movie came out and it can still surprise you. I first watched about two or three years ago and I very much liked it. What really surprised me is that even the bad guy, Dr. Varnick, made me laugh. This movie is funny and appropriate for children. It does contain some things that parents might consider inappropriate for children who are very little or who are sensitive. Some material might be frightening, but overall it's a nice movie. It will make you laugh, especially the dog-nappers.The Saint Bernard became the center of attention of the whole family.At the beginning I thought that the family wouldn't accept the dog, but I kept my hopes up. I would recommend this movie to anyon

### 2.2 Data Cleaning for movies

In [186]:
# create master dataset for movies
df_master_movies = df_movies.copy()

In [187]:
# Show the wrong ids identified earlier
invalid_ids = df_movies[~df_movies["movie_id"].str.match(r"^tt\d+$", na=False)]
invalid_ids

,movie_id,plot_summary,duration,genre,rating,release_date,plot_synopsis
1570,tt0104014/,"For a while now, beautiful 24-year-old Diana B...",1h 33min,"[Comedy, Drama]",5.3,1992-02-21,
1571,tt0114142/,"The marriage of David Burgess, a senior execut...",1h 32min,"[Drama, Thriller]",4.0,1999-01-29,


In [214]:
# Fix the movie id error identified earlier
df_master_movies["movie_id"] = df_master_movies["movie_id"].str.rstrip("/")

# Recheck
review_movie_ids = set(df["movie_id"])
movie_ids = set(df_master_movies["movie_id"])

# reviews without metadata
missing_movies = review_movie_ids - movie_ids

# metadata without reviews
unused_movies = movie_ids - review_movie_ids

print("Missing metadata:", len(missing_movies))
print("Unused metadata:", len(unused_movies))
# No more missing and unused metadata

Missing metadata: 0
Unused metadata: 0


In [190]:
# Changing release date from string to datetime
df_master_movies["release_date"] = pd.to_datetime(df_movies["release_date"], errors="coerce")

df_master_movies["release_year"] = df_master_movies["release_date"].dt.year

df_master_movies["release_year"].describe()
# Count of release years is 10 less than the number of movies, 10 movies do not havea release date

count    1562.000000
mean     2001.149808
std        13.575729
min      1921.000000
25%      1995.000000
50%      2003.000000
75%      2010.000000
max      2018.000000
Name: release_year, dtype: float64

In [191]:
# Checking release date for the 10 movies
df_master_movies[df_master_movies["release_year"].isna()][["movie_id", "release_date"]]

,movie_id,release_date
161,tt0060107,NaT
276,tt0099566,NaT
287,tt0104545,NaT
332,tt0052311,NaT
554,tt0050825,NaT
796,tt0100263,NaT
1208,tt0015864,NaT
1216,tt0050083,NaT
1472,tt0036868,NaT
1497,tt0047396,NaT


In [192]:
# Checking what the original release year entries were for these movies
failed_ids = df_master_movies[df_master_movies["release_year"].isna()]["movie_id"]

df_movies[df_movies["movie_id"].isin(failed_ids)][["movie_id", "release_date"]]

,movie_id,release_date
161,tt0060107,1973
276,tt0099566,1991-03
287,tt0104545,1994-02
332,tt0052311,1958-02
554,tt0050825,1957-11
796,tt0100263,1991-04
1208,tt0015864,1925
1216,tt0050083,1957-04
1472,tt0036868,1947
1497,tt0047396,1954-09


In [198]:
# Fix the empty release years
year_fix = {
    "tt0060107": 1973,
    "tt0099566": 1991,
    "tt0104545": 1994,
    "tt0052311": 1958,
    "tt0050825": 1957,
    "tt0100263": 1991,
    "tt0015864": 1925,
    "tt0050083": 1957,
    "tt0036868": 1947,
    "tt0047396": 1954,
}

df_master_movies["release_year"] = df_master_movies.apply(
    lambda row: year_fix.get(row["movie_id"], row["release_year"]),
    axis=1
)

# Check whether release years are fixed
print(df_master_movies[df_master_movies["release_year"].isna()][["movie_id", "release_date"]])
print(df_master_movies[df_master_movies["movie_id"].isin(failed_ids)][["movie_id", "release_year"]])
print(df_master_movies["release_year"].describe())

Empty DataFrame
Columns: [movie_id, release_date]
Index: []
       movie_id  release_year
161   tt0060107        1973.0
276   tt0099566        1991.0
287   tt0104545        1994.0
332   tt0052311        1958.0
554   tt0050825        1957.0
796   tt0100263        1991.0
1208  tt0015864        1925.0
1216  tt0050083        1957.0
1472  tt0036868        1947.0
1497  tt0047396        1954.0
count    1572.000000
mean     2000.917939
std        13.942056
min      1921.000000
25%      1995.000000
50%      2003.000000
75%      2010.000000
max      2018.000000
Name: release_year, dtype: float64


In [200]:
# Changing movie duration from string to number
df_master_movies["duration_minutes"] = (
    df_master_movies["duration"]
    .str.extract(r'(?:(\d+)h)?\s*(?:(\d+)min)?')
    .astype(float)
    .fillna(0)
    .assign(total=lambda x: x[0]*60 + x[1])["total"]
)

In [201]:
# Checking details of movie duration
df_master_movies[["duration", "duration_minutes"]].head()
df_master_movies["duration_minutes"].describe()

count    1572.000000
mean      115.269084
std        24.544471
min        42.000000
25%       100.000000
50%       113.000000
75%       128.000000
max       321.000000
Name: duration_minutes, dtype: float64

In [202]:
# Drop old string duration column
df_master_movies = df_master_movies.drop(columns=["duration"])

In [203]:
# Checking for outliers after data has been preprocessed
# Duration
print(len(df_master_movies[df_master_movies["duration_minutes"] > 300]))

# Rating
print(len(df_master_movies[(df_master_movies["rating"] < 0) | (df_master_movies["rating"] > 10)]))

# Release year
print(len(df_master_movies[df_master_movies["release_year"] < 1950]))

1
0
25


In [204]:
# Change name of movie rating from rating to movie_rating to avoid overlap with review rating
df_master_movies = df_master_movies.rename(columns={"rating": "movie_rating"})
# Check whether name change worked
print(df_master_movies.columns)

Index(['movie_id', 'plot_summary', 'genre', 'movie_rating', 'release_date',
       'plot_synopsis', 'release_year', 'duration_minutes'],
      dtype='object')


In [205]:
# Basic data cleaning function for movie summary and synopsis
def basic_clean_movie(text):
    if pd.isna(text):
        return ""
    text = str(text) # Forces text to be a String
    text = html.unescape(text) # Converts HTML to normal text
    text = unicodedata.normalize("NFKC", text) # Normalizes unicode into a consistent canonical form
    text = re.sub(r"<[^>]+>", " ", text) # Removes HTML tags 
    text = text.lower() # Converts all text to lowercase
    text = re.sub(r"\s+", " ", text) # Collapses all whitespace into a single space
    text = text.strip() # Removes whitespace in front and behind of text
    return text

In [206]:
# add cleaned summary and synopsis to dataframe
df_master_movies["plot_summary_clean"] = df_master_movies["plot_summary"].apply(basic_clean_movie)
df_master_movies["plot_synopsis_clean"] = df_master_movies["plot_synopsis"].apply(basic_clean_movie)

This following subsection is to check for any correlation between spoilers and movie-level details

In [207]:
# Create joined dataset
df_joined = df_master.merge(df_master_movies, on="movie_id", how="left")

In [222]:
# Compare means by class
df_joined.groupby("is_spoiler")[[
    "release_year",
    "duration_minutes",
    "movie_rating"
]].mean()

,release_year,duration_minutes,movie_rating
is_spoiler,,,
False,2002.459755,120.930968,7.297736
True,2004.661551,121.116438,7.279640


In [224]:
# Check correlation between spoiler rate and each movie-level detail
movie_level = df_joined.groupby("movie_id").agg({
    "is_spoiler": "mean",   # spoiler rate per movie
    "release_year": "first",
    "duration_minutes": "first",
    "movie_rating": "first"
})
movie_level.corr()
# Weak correlation between spoiler rate and movie-level details except for release year and spoiler

,is_spoiler,release_year,duration_minutes,movie_rating
is_spoiler,1.000000,0.277722,0.036884,0.026231
release_year,0.277722,1.000000,-0.057506,-0.199517
duration_minutes,0.036884,-0.057506,1.000000,0.259208
movie_rating,0.026231,-0.199517,0.259208,1.000000


In [210]:
movie_level.groupby(pd.cut(movie_level["release_year"], bins=10))["is_spoiler"].mean()
# the correlation observed between release year and spoiler boolean is likely due to general
# increased review counts for modern movies, not very useful for prediction of spoilers

C:\Users\brchu\AppData\Local\Temp\ipykernel_13128\2545046363.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  movie_level.groupby(pd.cut(movie_level["release_year"], bins=10))["is_spoiler"].mean()


release_year
(1920.903, 1930.7]    0.260148
(1930.7, 1940.4]      0.263170
(1940.4, 1950.1]      0.247975
(1950.1, 1959.8]      0.264974
(1959.8, 1969.5]      0.238448
(1969.5, 1979.2]      0.235035
(1979.2, 1988.9]      0.230964
(1988.9, 1998.6]      0.208278
(1998.6, 2008.3]      0.216609
(2008.3, 2018.0]      0.318832
Name: is_spoiler, dtype: float64

In [213]:
# Code for getting the json of df_master_movies
df_master_movies.to_json("movie_details_cleaned.json", orient="records", force_ascii=False)

## 3. Data Splitting
Split data by movies so that the model doesnt try to learn movie specific context to detect spoilers.  
Train/test split is group-based (by `movie_id`) and **CV is Stratified Group K-Fold** on the training portion.  
Approximate Split Proportion:  
Training: 0.80  
Testing: 0.20

In [215]:
# Function for splitting data into training and testing sets
def group_data_split(df, group_col="movie_id", test_size=0.2, random_state=42):
    """
    Split a dataframe into train and test sets using group-based splitting.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    group_col : str, default="movie_id"
        Column used to define groups that must not be split across train and test.
    test_size : float, default=0.2
        Proportion of groups to place in the test set.
    random_state : int, default=42
        Random seed for reproducibility.

    Returns
    -------
    df_train : pandas.DataFrame
        Training subset.
    df_test : pandas.DataFrame
        Test subset.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)

    train_idx, test_idx = next(gss.split(df, groups=df[group_col]))

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)

    return df_train, df_test

In [216]:
# Splitting data into training set and testing set (grouped by movie_id)
df_train, df_test = group_data_split(
    df_master, 
    group_col="movie_id", 
    test_size=0.2, 
    random_state=42
)

In [217]:
# Create Stratified Group K-Fold splits on the training set
N_SPLITS = 5
CV_SEED = 42
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)

cv_splits = list(
    cv.split(
        df_train["clean_text"],
        df_train["is_spoiler"],
        groups=df_train["movie_id"],
    )
)
print("Number of CV folds:", len(cv_splits))

Number of CV folds: 5


In [218]:
# Sanity checks for data splitting
# No leakage between train and test
assert set(df_train["movie_id"]).isdisjoint(df_test["movie_id"])

# Check approximate sizes
print("Train size:", len(df_train))
print("Test size:", len(df_test))
print("Train proportion:", len(df_train) / len(df_master))
print("Test proportion:", len(df_test) / len(df_master))

# Check class balance
print("\nSpoiler rate")
print("Full  :", df_master["is_spoiler"].mean())
print("Train :", df_train["is_spoiler"].mean())
print("Test  :", df_test["is_spoiler"].mean())

# Check number of unique movies
print("\nUnique movies")
print("Full  :", df_master["movie_id"].nunique())
print("Train :", df_train["movie_id"].nunique())
print("Test  :", df_test["movie_id"].nunique())

# Check fold sizes and balance
for i, (train_idx, val_idx) in enumerate(cv_splits, 1):
    y_tr = df_train.iloc[train_idx]["is_spoiler"]
    y_val = df_train.iloc[val_idx]["is_spoiler"]
    print(f"Fold {i}: train={len(train_idx)}, val={len(val_idx)}, train_spoiler={y_tr.mean():.3f}, val_spoiler={y_val.mean():.3f}")

Train size: 463816
Test size: 109463
Train proportion: 0.8090580677122309
Test proportion: 0.19094193228776912

Spoiler rate
Full  : 0.26299062062276835
Train : 0.2641262914604067
Test  : 0.2581785626193326

Unique movies
Full  : 1572
Train : 1257
Test  : 315
Fold 1: train=363883, val=99933, train_spoiler=0.264, val_spoiler=0.265
Fold 2: train=368380, val=95436, train_spoiler=0.264, val_spoiler=0.264
Fold 3: train=374197, val=89619, train_spoiler=0.267, val_spoiler=0.254
Fold 4: train=374897, val=88919, train_spoiler=0.263, val_spoiler=0.271
Fold 5: train=373907, val=89909, train_spoiler=0.263, val_spoiler=0.267


In [221]:
# Checking movie level features across folds
for i, (_, val_idx) in enumerate(cv_splits):
    fold_movies = df_train.iloc[val_idx]["movie_id"].unique()
    fold_df = df_master_movies[df_master_movies["movie_id"].isin(fold_movies)]
    genre_counts = fold_df["genre"].explode().value_counts(normalize=True)
    
    print(f"\nFold {i}")
    print("Movies:", len(fold_movies))
    print("Avg movie rating:", fold_df["movie_rating"].mean())
    print("Avg duration:", fold_df["duration_minutes"].mean())
    print("Avg year:", fold_df["release_year"].mean())
    print("Top genres:", genre_counts.head())

# Distribution of movie data is relatively even


Fold 0
Movies: 252
Avg movie rating: 7.166666666666667
Avg duration: 116.62698412698413
Avg year: 2000.5119047619048
Top genres: genre
Drama        0.189189
Comedy       0.135135
Action       0.117117
Adventure    0.111111
Crime        0.075075
Name: proportion, dtype: float64

Fold 1
Movies: 257
Avg movie rating: 7.038132295719844
Avg duration: 114.34630350194553
Avg year: 2000.9922178988327
Top genres: genre
Drama        0.174174
Comedy       0.127628
Action       0.115616
Adventure    0.102102
Crime        0.078078
Name: proportion, dtype: float64

Fold 2
Movies: 246
Avg movie rating: 7.123983739837398
Avg duration: 114.91056910569105
Avg year: 2000.4024390243903
Top genres: genre
Drama        0.195925
Comedy       0.120690
Adventure    0.109718
Action       0.090909
Crime        0.064263
Name: proportion, dtype: float64

Fold 3
Movies: 250
Avg movie rating: 7.022
Avg duration: 114.92
Avg year: 2001.42
Top genres: genre
Drama        0.196947
Comedy       0.125191
Adventure    0.109

## 4. Feature Engineering

This section builds three feature sets from the fixed train, validation, and test splits: TF-IDF, TF-IDF + lemmatization, and BERT embeddings. All artifacts are saved to disk for reuse in later model training and comparison, with grouping controlled by `movie_id`.

### 4.1 TF-IDF (No Lemmatization)

In [ ]:
# TF-IDF (no lemma) for CV folds and full train/test
from sklearn.feature_extraction.text import TfidfVectorizer

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = "clean_text"
LABEL_COL = "is_spoiler"
GROUP_COL = "movie_id"

def clean_corpus(texts):
    if TEXT_COL == "clean_text":
        return [str(t) for t in texts]
    return [basic_clean(t) for t in texts]

def make_vectorizer():
    # float32 saves memory and is usually fine for TF-IDF.
    return TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english",
        dtype=np.float32,
    )

def fit_transform_vectorizer(texts):
    vec = make_vectorizer()
    X = vec.fit_transform(clean_corpus(texts))
    return vec, X

def transform_vectorizer(texts, vec):
    return vec.transform(clean_corpus(texts))

def build_cv_fold_tfidf(df_train, fold_idx, train_idx, val_idx):
    fold_tag = f"fold{fold_idx}"
    df_tr = df_train.iloc[train_idx].reset_index(drop=True)
    df_val = df_train.iloc[val_idx].reset_index(drop=True)

    # Labels for this fold
    np.save(ARTIFACT_DIR / f"labels_train_{fold_tag}.npy", df_tr[LABEL_COL].to_numpy(dtype=np.int64))
    np.save(ARTIFACT_DIR / f"labels_val_{fold_tag}.npy", df_val[LABEL_COL].to_numpy(dtype=np.int64))

    # TF-IDF (fit+transform once for train; only transform for val)
    tfidf_vec, X_tr = fit_transform_vectorizer(df_tr[TEXT_COL])
    X_val = transform_vectorizer(df_val[TEXT_COL], tfidf_vec)

    joblib.dump(tfidf_vec, ARTIFACT_DIR / f"tfidf_vectorizer_{fold_tag}.joblib")
    sparse.save_npz(ARTIFACT_DIR / f"X_tfidf_train_{fold_tag}.npz", X_tr)
    sparse.save_npz(ARTIFACT_DIR / f"X_tfidf_val_{fold_tag}.npz", X_val)

def build_full_train_test_tfidf(df_train, df_test):
    # Labels
    np.save(ARTIFACT_DIR / "labels_train_full.npy", df_train[LABEL_COL].to_numpy(dtype=np.int64))
    np.save(ARTIFACT_DIR / "labels_test.npy", df_test[LABEL_COL].to_numpy(dtype=np.int64))

    # TF-IDF (fit+transform once on full training set)
    tfidf_vec, X_train = fit_transform_vectorizer(df_train[TEXT_COL])
    X_test = transform_vectorizer(df_test[TEXT_COL], tfidf_vec)

    joblib.dump(tfidf_vec, ARTIFACT_DIR / "tfidf_vectorizer_full.joblib")
    sparse.save_npz(ARTIFACT_DIR / "X_tfidf_train_full.npz", X_train)
    sparse.save_npz(ARTIFACT_DIR / "X_tfidf_test.npz", X_test)

def build_cv_tfidf_features(df_train, df_test, cv_splits):
    for i, (train_idx, val_idx) in enumerate(cv_splits, 1):
        print(f"[tfidf] fold {i}/{len(cv_splits)}")
        build_cv_fold_tfidf(df_train, i, train_idx, val_idx)
    print("[tfidf] full train/test")
    build_full_train_test_tfidf(df_train, df_test)

build_cv_tfidf_features(df_train, df_test, cv_splits)

### 4.2 TF-IDF + Lemmatization

In [ ]:
# TF-IDF + lemma for CV folds and full train/test
# NOTE: The main bottleneck is lemmatization. This version lemmatizes ONCE (cached)
# and reuses the lemmatized strings across all CV folds/transforms.
from sklearn.feature_extraction.text import TfidfVectorizer

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse ARTIFACT_DIR / TEXT_COL / LABEL_COL / df_train / df_test / cv_splits defined above.
# Provide a small fallback so this cell is still runnable if executed out-of-order.
try:
    clean_corpus  # noqa: B018
except NameError:
    def clean_corpus(texts):
        return [str(t) for t in texts]

CACHE_DIR = ARTIFACT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def ensure_nltk_resources():
    import nltk
    # Only required for WordNetLemmatizer.
    resources = ["wordnet", "omw-1.4"]
    for res in resources:
        try:
            nltk.data.find(f"corpora/{res}")
        except LookupError:
            nltk.download(res, quiet=True)
    return nltk

nltk = ensure_nltk_resources()
from nltk.stem import WordNetLemmatizer

_TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def _lemmatize_batch(text_batch):
    lem = WordNetLemmatizer()
    out = []
    for text in text_batch:
        tokens = _TOKEN_RE.findall(str(text))
        out.append(" ".join(lem.lemmatize(tok) for tok in tokens))
    return out

def lemmatize_texts_cached(texts, cache_path: Path, n_jobs: int = -1, batch_size: int = 2000):
    """Return np.ndarray[str] of lemmatized texts, cached to disk."""
    if cache_path.exists():
        print(f"[lemma] cache hit: {cache_path}")
        return joblib.load(cache_path)

    print(f"[lemma] building cache: {cache_path} (this can take a while)")

    # Clean first (handles NaNs, non-strings)
    cleaned = clean_corpus(texts)

    # Batch to avoid joblib overhead per document (important for large corpora)
    batches = [cleaned[i : i + batch_size] for i in range(0, len(cleaned), batch_size)]

    from joblib import Parallel, delayed

    parts = Parallel(n_jobs=n_jobs, prefer="processes")(
        delayed(_lemmatize_batch)(b) for b in batches
    )

    # Flatten list-of-lists
    lemmatized = np.asarray([t for part in parts for t in part], dtype=object)

    joblib.dump(lemmatized, cache_path, compress=3)
    print(f"[lemma] saved cache: {cache_path}")
    return lemmatized

# Precompute lemma text ONCE for full train/test (reused for every fold)
lemma_train = lemmatize_texts_cached(
    df_train[TEXT_COL],
    CACHE_DIR / f"lemma_train_{TEXT_COL}.joblib",
)
lemma_test = lemmatize_texts_cached(
    df_test[TEXT_COL],
    CACHE_DIR / f"lemma_test_{TEXT_COL}.joblib",
)

def make_lemma_vectorizer():
    # dtype float32 to reduce memory; keep same settings as base TF-IDF.
    return TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english",
        dtype=np.float32,
    )

def build_cv_fold_tfidf_lemma(fold_idx, train_idx, val_idx):
    fold_tag = f"fold{fold_idx}"

    vec = make_lemma_vectorizer()

    X_tr_l = vec.fit_transform(lemma_train[train_idx])
    X_val_l = vec.transform(lemma_train[val_idx])

    joblib.dump(vec, ARTIFACT_DIR / f"tfidf_lemma_vectorizer_{fold_tag}.joblib")
    sparse.save_npz(ARTIFACT_DIR / f"X_tfidf_lemma_train_{fold_tag}.npz", X_tr_l)
    sparse.save_npz(ARTIFACT_DIR / f"X_tfidf_lemma_val_{fold_tag}.npz", X_val_l)

def build_full_train_test_tfidf_lemma():
    vec = make_lemma_vectorizer()

    X_train_l = vec.fit_transform(lemma_train)
    X_test_l = vec.transform(lemma_test)

    joblib.dump(vec, ARTIFACT_DIR / "tfidf_lemma_vectorizer_full.joblib")
    sparse.save_npz(ARTIFACT_DIR / "X_tfidf_lemma_train_full.npz", X_train_l)
    sparse.save_npz(ARTIFACT_DIR / "X_tfidf_lemma_test.npz", X_test_l)

def build_cv_tfidf_lemma_features(cv_splits):
    for i, (train_idx, val_idx) in enumerate(cv_splits, 1):
        print(f"[tfidf+lemma] fold {i}/{len(cv_splits)}")
        build_cv_fold_tfidf_lemma(i, train_idx, val_idx)
    print("[tfidf+lemma] full train/test")
    build_full_train_test_tfidf_lemma()

build_cv_tfidf_lemma_features(cv_splits)

### 4.3 BERT Embeddings

In [ ]:
# BERT embeddings (single train/test, no CV folds)
from sentence_transformers import SentenceTransformer
import time

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

BERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BERT_BATCH_SIZE = 64

# Function to generate and save BERT embeddings for a dataframe
def save_bert_embeddings(texts, artifact_path, bert_model, batch_size=64):
    start_time = time.perf_counter()
    X = bert_model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    np.save(artifact_path, X)
    print(f"Saved BERT embeddings to {artifact_path} (shape: {X.shape}, time: {time.perf_counter() - start_time:.1f}s)")

bert_model = SentenceTransformer(BERT_MODEL_NAME)

# Save BERT embeddings for full train and test splits
save_bert_embeddings(df_train["clean_text"], ARTIFACT_DIR / "X_bert_train_full.npy", bert_model, BERT_BATCH_SIZE)
np.save(ARTIFACT_DIR / "bert_train_ids.npy", df_train.index.to_numpy()) # Save review IDs for train (for debugging, does not affect main workflow)

save_bert_embeddings(df_test["clean_text"], ARTIFACT_DIR / "X_bert_test.npy", bert_model, BERT_BATCH_SIZE)
np.save(ARTIFACT_DIR / "bert_test_ids.npy", df_test.index.to_numpy()) # Save review IDs for test (for debugging, does not affect main workflow)


### 4.4 Artifact Summary

Each cross-validation fold, such as `fold1`, produces:
- `labels_train_fold1.npy`, `labels_val_fold1.npy`
- `X_tfidf_train_fold1.npz`, `X_tfidf_val_fold1.npz` and `tfidf_vectorizer_fold1.joblib`
- `X_tfidf_lemma_train_fold1.npz`, `X_tfidf_lemma_val_fold1.npz` and `tfidf_lemma_vectorizer_fold1.joblib`

After the cross-validation artifacts are created, the notebook also saves full train/test artifacts for final model fitting:
- `labels_train_full.npy`, `labels_test.npy`
- `X_tfidf_train_full.npz`, `X_tfidf_test.npz` and `tfidf_vectorizer_full.joblib`
- `X_tfidf_lemma_train_full.npz`, `X_tfidf_lemma_test.npz` and `tfidf_lemma_vectorizer_full.joblib`
- `X_bert_train_full.npy`, `X_bert_test.npy`

Lemma preprocessing also writes cached text files to speed up repeated runs:
- `cache/lemma_train_clean_text.joblib`
- `cache/lemma_test_clean_text.joblib`

All generated files are stored under `artifacts`.

### 4.5 Feature Artifact Loading

Run the next cell to load the saved train, validation, and test artifacts from disk so later sections can reuse exactly the same prepared inputs.

In [34]:
# Load CV artifacts into memory for downstream model training
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5

def load_cv_fold(fold_idx):
    fold_tag = f"fold{fold_idx}"
    artifacts = {}
    artifacts["y_train"] = np.load(ARTIFACT_DIR / f"labels_train_{fold_tag}.npy")
    artifacts["y_val"] = np.load(ARTIFACT_DIR / f"labels_val_{fold_tag}.npy")
    artifacts["X_tfidf_train"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_train_{fold_tag}.npz")
    artifacts["X_tfidf_val"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_val_{fold_tag}.npz")
    # artifacts["tfidf_vectorizer"] = joblib.load(ARTIFACT_DIR / f"tfidf_vectorizer_{fold_tag}.joblib")
    artifacts["X_tfidf_lemma_train"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_lemma_train_{fold_tag}.npz")
    artifacts["X_tfidf_lemma_val"] = sparse.load_npz(ARTIFACT_DIR / f"X_tfidf_lemma_val_{fold_tag}.npz")
    # artifacts["tfidf_lemma_vectorizer"] = joblib.load(ARTIFACT_DIR / f"tfidf_lemma_vectorizer_{fold_tag}.joblib")
    return artifacts

def load_full_train_test():
    artifacts = {}
    artifacts["y_train_full"] = np.load(ARTIFACT_DIR / "labels_train_full.npy")
    artifacts["y_test"] = np.load(ARTIFACT_DIR / "labels_test.npy")
    artifacts["X_tfidf_train_full"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_train_full.npz")
    artifacts["X_tfidf_test"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_test.npz")
    # artifacts["tfidf_vectorizer_full"] = joblib.load(ARTIFACT_DIR / "tfidf_vectorizer_full.joblib")
    artifacts["X_tfidf_lemma_train_full"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_lemma_train_full.npz")
    artifacts["X_tfidf_lemma_test"] = sparse.load_npz(ARTIFACT_DIR / "X_tfidf_lemma_test.npz")
    # artifacts["tfidf_lemma_vectorizer_full"] = joblib.load(ARTIFACT_DIR / "tfidf_lemma_vectorizer_full.joblib")
    artifacts["X_bert_train_full"] = np.load(ARTIFACT_DIR / "X_bert_train_full.npy")
    artifacts["X_bert_test"] = np.load(ARTIFACT_DIR / "X_bert_test.npy")
    return artifacts

cv_folds = {f"fold{i}": load_cv_fold(i) for i in range(1, N_SPLITS + 1)}
full_train_test = load_full_train_test()
print("Loaded CV folds:", list(cv_folds.keys()))
print("Loaded full train/test:", list(full_train_test.keys()))

Loaded CV folds: ['fold1', 'fold2', 'fold3', 'fold4', 'fold5']
Loaded full train/test: ['y_train_full', 'y_test', 'X_tfidf_train_full', 'X_tfidf_test', 'X_tfidf_lemma_train_full', 'X_tfidf_lemma_test', 'X_bert_train_full', 'X_bert_test']


## 5. Model Training and Hyperparameter Tuning

This section uses a fixed model-selection protocol to keep comparisons fair and avoid leakage.

- `df_train` is used for cross-validation and final model fitting.
- `cv_splits` contains the fixed folds reused across feature engineering methods.
- `df_test` is reserved for one-time final evaluation only.

For each model family, the workflow is:
1. Run cross-validation on `df_train` to select hyperparameters.
2. Refit the model on the full training set.
3. Evaluate once on `df_test` in Section 6.

### 5.1 Shared Tuning Settings

All models in Section 5 select hyperparameters by mean validation AUC using the same cross-validation protocol. The search ranges are defined once below for consistency, while feature-specific implementation details such as sparse versus dense preprocessing remain inside the individual model cells.

This subsection contains the shared scoring, fold-count, random seed, and search ranges used during model selection. The fast-rerun fixed hyperparameters are separated into the next subsection.

In [62]:
SCORING_METRIC = "roc_auc"
N_SPLITS = 5
RANDOM_STATE = 42
CLASS_WEIGHT = "balanced"

LR_C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
BERT_LR_C_GRID = [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
LR_MAX_ITER = 2000

SVM_C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
SVM_TOL = 1e-4
SVM_MAX_ITER = 3000
SVM_CALIBRATION_CV = 3
SVM_CALIBRATION_METHOD = "sigmoid"

DT_PARAM_GRID = {
    "max_depth": [5, 10, 15],
    "min_samples_split": [10, 25, 50],
    "min_samples_leaf": [5, 10, 20],
    "ccp_alpha": [0.0, 1e-4, 1e-3, 1e-2],
}
DT_RANDOM_SEARCH_ITER = 10

### 5.2 Fast Rerun Settings

This subsection stores fixed hyperparameters from earlier CV results so repeated runs can skip tuning and go straight to refitting on the full training split.

Set `USE_FIXED_BEST_PARAMS = True` to use the saved values below when they are available. Leave a value as `None` if that model has not completed its first tuning run yet.

In [63]:
USE_FIXED_BEST_PARAMS = True

LR_TFIDF_FIXED_C = 0.3
LR_TFIDF_LEMMA_FIXED_C = 0.3
LR_BERT_FIXED_C = 10.0
SVM_TFIDF_FIXED_C = 0.03
SVM_TFIDF_LEMMA_FIXED_C = 0.03
SVM_BERT_FIXED_C = 3.0
DT_TFIDF_FIXED_PARAMS = {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001}
DT_TFIDF_LEMMA_FIXED_PARAMS = {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001}
DT_BERT_FIXED_PARAMS = {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001}

### 5.3 Logistic Regression

#### 5.3.1 TF-IDF

In [64]:
from sklearn.linear_model import LogisticRegression

if USE_FIXED_BEST_PARAMS and LR_TFIDF_FIXED_C is not None:
    best_c = LR_TFIDF_FIXED_C
    fold_rows = []
    for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"LR TF-IDF fixed C={best_c:g}"):
        artifacts = load_cv_fold(fold_idx)
        X_train = artifacts["X_tfidf_train"]
        X_val = artifacts["X_tfidf_val"]
        y_train = artifacts["y_train"]
        y_val = artifacts["y_val"]

        model = LogisticRegression(
            C=best_c,
            solver="saga",
            max_iter=LR_MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details = pd.DataFrame(fold_rows)
    fixed_cv_summary = fixed_cv_fold_details[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed C: {best_c} (full search skipped)")
    print("Fold-by-fold metrics for fixed C:")
    display(fixed_cv_fold_details.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed C:")
    display(fixed_cv_summary.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows = []
    cv_fold_details_by_c = {}

    for c_value in tqdm(LR_C_GRID, desc="LR TF-IDF C search"):
        fold_rows = []

        for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"CV folds (C={c_value:g})", leave=False):
            artifacts = load_cv_fold(fold_idx)
            X_train = artifacts["X_tfidf_train"]
            X_val = artifacts["X_tfidf_val"]
            y_train = artifacts["y_train"]
            y_val = artifacts["y_val"]

            model = LogisticRegression(
                C=c_value,
                solver="saga",
                max_iter=LR_MAX_ITER,
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            )
            model.fit(X_train, y_train)

            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)

        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_c[c_value] = fold_df

        cv_summary_rows.append(
            {
                "C": c_value,
                "mean_auc": fold_df["auc"].mean(),
                "std_auc": fold_df["auc"].std(),
                "mean_average_precision": fold_df["average_precision"].mean(),
                "mean_f1": fold_df["f1"].mean(),
                "mean_f2": fold_df["f2"].mean(),
                "mean_precision": fold_df["precision"].mean(),
                "mean_recall": fold_df["recall"].mean(),
                "mean_accuracy": fold_df["accuracy"].mean(),
            }
        )

    cv_results = pd.DataFrame(cv_summary_rows).sort_values("mean_auc", ascending=False).reset_index(drop=True)

    print("CV summary across C values (ranked by mean AUC):")
    display(cv_results.rename(columns={"mean_average_precision": "mean_pr-auc"}))

    best_c = float(cv_results.loc[0, "C"])
    print(f"\nBest C by CV mean AUC: {best_c}")
    print("Fold-by-fold metrics for best C:")
    display(cv_fold_details_by_c[best_c].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary = cv_fold_details_by_c[best_c][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best C:")
    display(best_cv_summary.rename(columns={"average_precision": "pr-auc"}))

artifacts = load_full_train_test()
X_train_full = artifacts["X_tfidf_train_full"]
y_train_full = artifacts["y_train_full"]

final_model = LogisticRegression(
    C=best_c,
    solver="saga",
    max_iter=LR_MAX_ITER,
    random_state=RANDOM_STATE,
    class_weight=CLASS_WEIGHT,
)
final_model.fit(X_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
logreg_tfidf_best_c = best_c
logreg_tfidf_model = final_model

LR TF-IDF fixed C=0.3:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed C: 0.3 (full search skipped)
Fold-by-fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.765548,0.566527,0.549676,0.601869,0.480263,0.642544,0.721577,1
1,0.768599,0.570017,0.551374,0.598118,0.487833,0.633947,0.726907,2
2,0.763128,0.563197,0.546439,0.598066,0.477710,0.638268,0.719633,3
3,0.763943,0.557233,0.545101,0.600318,0.472645,0.643795,0.716731,4
4,0.763487,0.565164,0.546060,0.593441,0.481930,0.629878,0.724350,5


Average fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.764941,0.564428,0.54773,0.598363,0.480076,0.637686,0.72184


#### 5.3.2 TF-IDF + Lemmatization

In [65]:
from sklearn.linear_model import LogisticRegression

if USE_FIXED_BEST_PARAMS and LR_TFIDF_LEMMA_FIXED_C is not None:
    best_c_lemma = LR_TFIDF_LEMMA_FIXED_C
    fold_rows = []
    for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"LR TF-IDF+Lemma fixed C={best_c_lemma:g}"):
        artifacts = load_cv_fold(fold_idx)
        X_train = artifacts["X_tfidf_lemma_train"]
        X_val = artifacts["X_tfidf_lemma_val"]
        y_train = artifacts["y_train"]
        y_val = artifacts["y_val"]

        model = LogisticRegression(
            C=best_c_lemma,
            solver="saga",
            max_iter=LR_MAX_ITER,
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)

        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_lemma = pd.DataFrame(fold_rows)
    fixed_cv_summary_lemma = fixed_cv_fold_details_lemma[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed C: {best_c_lemma} (full search skipped)")
    print("Fold-by-fold metrics for fixed C:")
    display(fixed_cv_fold_details_lemma.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed C:")
    display(fixed_cv_summary_lemma.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_lemma = []
    cv_fold_details_by_c_lemma = {}

    for c_value in tqdm(LR_C_GRID, desc="LR TF-IDF+Lemma C search"):
        fold_rows = []

        for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"CV folds (C={c_value:g})", leave=False):
            artifacts = load_cv_fold(fold_idx)
            X_train = artifacts["X_tfidf_lemma_train"]
            X_val = artifacts["X_tfidf_lemma_val"]
            y_train = artifacts["y_train"]
            y_val = artifacts["y_val"]

            model = LogisticRegression(
                C=c_value,
                solver="saga",
                max_iter=LR_MAX_ITER,
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            )
            model.fit(X_train, y_train)

            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)

        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_c_lemma[c_value] = fold_df

        cv_summary_rows_lemma.append(
            {
                "C": c_value,
                "mean_auc": fold_df["auc"].mean(),
                "std_auc": fold_df["auc"].std(),
                "mean_average_precision": fold_df["average_precision"].mean(),
                "mean_f1": fold_df["f1"].mean(),
                "mean_f2": fold_df["f2"].mean(),
                "mean_precision": fold_df["precision"].mean(),
                "mean_recall": fold_df["recall"].mean(),
                "mean_accuracy": fold_df["accuracy"].mean(),
            }
        )

    cv_results_lemma = pd.DataFrame(cv_summary_rows_lemma).sort_values("mean_auc", ascending=False).reset_index(drop=True)

    print("CV summary across C values (ranked by mean AUC):")
    display(cv_results_lemma.rename(columns={"mean_average_precision": "mean_pr-auc"}))

    best_c_lemma = float(cv_results_lemma.loc[0, "C"])
    print(f"\nBest C by CV mean AUC: {best_c_lemma}")
    print("Fold-by-fold metrics for best C:")
    display(cv_fold_details_by_c_lemma[best_c_lemma].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_lemma = cv_fold_details_by_c_lemma[best_c_lemma][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best C:")
    display(best_cv_summary_lemma.rename(columns={"average_precision": "pr-auc"}))

artifacts = load_full_train_test()
X_train_full = artifacts["X_tfidf_lemma_train_full"]
y_train_full = artifacts["y_train_full"]

final_model_lemma = LogisticRegression(
    C=best_c_lemma,
    solver="saga",
    max_iter=LR_MAX_ITER,
    random_state=RANDOM_STATE,
    class_weight=CLASS_WEIGHT,
)
final_model_lemma.fit(X_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
logreg_tfidf_lemma_best_c = best_c_lemma
logreg_tfidf_lemma_model = final_model_lemma

LR TF-IDF+Lemma fixed C=0.3:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed C: 0.3 (full search skipped)
Fold-by-fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.764337,0.565185,0.548074,0.600737,0.478206,0.641852,0.720072,1
1,0.768082,0.569744,0.550900,0.597526,0.487500,0.633256,0.726681,2
2,0.762648,0.563275,0.546018,0.597565,0.477386,0.637699,0.719407,3
3,0.764031,0.558172,0.545205,0.600047,0.473133,0.643179,0.717120,4
4,0.762964,0.565329,0.544952,0.593724,0.479326,0.631397,0.722447,5


Average fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.764412,0.564341,0.54703,0.59792,0.47911,0.637477,0.721145


#### 5.3.3 BERT Embeddings

In [66]:
from sklearn.linear_model import LogisticRegression

artifacts = load_full_train_test()
X_bert_train_full = artifacts["X_bert_train_full"]
y_train_full = artifacts["y_train_full"]

if USE_FIXED_BEST_PARAMS and LR_BERT_FIXED_C is not None:
    best_c_bert = LR_BERT_FIXED_C
    fold_rows = []
    for fold_idx, (train_idx, val_idx) in tqdm(
        enumerate(cv_splits, 1),
        total=len(cv_splits),
        desc=f"LR BERT fixed C={best_c_bert:g}",
    ):
        X_train = X_bert_train_full[train_idx]
        X_val = X_bert_train_full[val_idx]
        y_train = y_train_full[train_idx]
        y_val = y_train_full[val_idx]
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=best_c_bert,
                solver="lbfgs",
                max_iter=LR_MAX_ITER,
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            ),
        )
        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_bert = pd.DataFrame(fold_rows)
    fixed_cv_summary_bert = fixed_cv_fold_details_bert[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed C: {best_c_bert} (full search skipped)")
    print("Fold-by-fold metrics for fixed C:")
    display(fixed_cv_fold_details_bert.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed C:")
    display(fixed_cv_summary_bert.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_bert = []
    cv_fold_details_by_c_bert = {}

    for c_value in tqdm(BERT_LR_C_GRID, desc="LR BERT C search"):
        fold_rows = []
        for fold_idx, (train_idx, val_idx) in tqdm(
            enumerate(cv_splits, 1),
            total=len(cv_splits),
            desc=f"CV folds (C={c_value:g})",
            leave=False,
        ):
            X_train = X_bert_train_full[train_idx]
            X_val = X_bert_train_full[val_idx]
            y_train = y_train_full[train_idx]
            y_val = y_train_full[val_idx]
            model = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    C=c_value,
                    solver="lbfgs",
                    max_iter=LR_MAX_ITER,
                    random_state=RANDOM_STATE,
                    class_weight=CLASS_WEIGHT,
                ),
            )
            model.fit(X_train, y_train)
            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)

        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_c_bert[c_value] = fold_df
        cv_summary_rows_bert.append({
            "C": c_value,
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_average_precision": fold_df["average_precision"].mean(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_f2": fold_df["f2"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        })

    cv_results_bert = pd.DataFrame(cv_summary_rows_bert).sort_values("mean_auc", ascending=False).reset_index(drop=True)

    print("CV summary across C values (ranked by mean AUC):")
    display(cv_results_bert.rename(columns={"mean_average_precision": "mean_pr-auc"}))

    best_c_bert = float(cv_results_bert.loc[0, "C"])
    print(f"\nBest C by CV mean AUC: {best_c_bert}")
    print("Fold-by-fold metrics for best C:")
    display(cv_fold_details_by_c_bert[best_c_bert].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_bert = cv_fold_details_by_c_bert[best_c_bert][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best C:")
    display(best_cv_summary_bert.rename(columns={"average_precision": "pr-auc"}))

final_model_bert = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=best_c_bert,
        solver="lbfgs",
        max_iter=LR_MAX_ITER,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    ),
)
final_model_bert.fit(X_bert_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
logreg_bert_best_c = best_c_bert
logreg_bert_model = final_model_bert

LR BERT fixed C=10:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed C: 10.0 (full search skipped)
Fold-by-fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.722909,0.480866,0.505411,0.578538,0.417466,0.640300,0.667547,1
1,0.731586,0.495878,0.512472,0.599663,0.412508,0.676382,0.660862,2
2,0.729719,0.477715,0.501678,0.584471,0.405858,0.656725,0.668575,3
3,0.720772,0.485877,0.511091,0.600185,0.409723,0.679107,0.648017,4
4,0.720596,0.478631,0.507524,0.588229,0.413069,0.657982,0.659278,5


Average fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.725116,0.483794,0.507635,0.590217,0.411725,0.662099,0.660856


### 5.4 Support Vector Machine

#### 5.4.1 TF-IDF

In [67]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

if USE_FIXED_BEST_PARAMS and SVM_TFIDF_FIXED_C is not None:
    best_c_svm = SVM_TFIDF_FIXED_C
    fold_rows = []
    for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"SVM TF-IDF fixed C={best_c_svm:g}"):
        artifacts = load_cv_fold(fold_idx)
        X_train = artifacts["X_tfidf_train"]
        X_val = artifacts["X_tfidf_val"]
        y_train = artifacts["y_train"]
        y_val = artifacts["y_val"]
        svm = LinearSVC(C=best_c_svm, tol=SVM_TOL, max_iter=SVM_MAX_ITER, dual="auto", random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)
        model = CalibratedClassifierCV(svm, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_svm = pd.DataFrame(fold_rows)
    fixed_cv_summary_svm = fixed_cv_fold_details_svm[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed C: {best_c_svm} (full search skipped)")
    print("Fold-by-fold metrics for fixed C:")
    display(fixed_cv_fold_details_svm.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed C:")
    display(fixed_cv_summary_svm.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_svm = []
    cv_fold_details_by_c_svm = {}

    for c_value in tqdm(SVM_C_GRID, desc="SVM TF-IDF C search"):
        fold_rows = []
        for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"CV folds (C={c_value:g})", leave=False):
            artifacts = load_cv_fold(fold_idx)
            X_train = artifacts["X_tfidf_train"]
            X_val = artifacts["X_tfidf_val"]
            y_train = artifacts["y_train"]
            y_val = artifacts["y_val"]
            svm = LinearSVC(C=c_value, tol=SVM_TOL, max_iter=SVM_MAX_ITER, dual="auto", random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)
            model = CalibratedClassifierCV(svm, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
            model.fit(X_train, y_train)
            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)
        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_c_svm[c_value] = fold_df
        cv_summary_rows_svm.append({
            "C": c_value,
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_average_precision": fold_df["average_precision"].mean(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_f2": fold_df["f2"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        })

    cv_results_svm = pd.DataFrame(cv_summary_rows_svm).sort_values("mean_auc", ascending=False).reset_index(drop=True)
    print("CV summary across C values (ranked by mean AUC):")
    display(cv_results_svm.rename(columns={"mean_average_precision": "mean_pr-auc"}))
    best_c_svm = float(cv_results_svm.loc[0, "C"])
    print(f"\nBest C by CV mean AUC: {best_c_svm}")
    print("Fold-by-fold metrics for best C:")
    display(cv_fold_details_by_c_svm[best_c_svm].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_svm = cv_fold_details_by_c_svm[best_c_svm][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best C:")
    display(best_cv_summary_svm.rename(columns={"average_precision": "pr-auc"}))

artifacts = load_full_train_test()
X_train_full = artifacts["X_tfidf_train_full"]
y_train_full = artifacts["y_train_full"]
final_svm = LinearSVC(C=best_c_svm, tol=SVM_TOL, max_iter=SVM_MAX_ITER, dual="auto", random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)
final_model_svm = CalibratedClassifierCV(final_svm, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
final_model_svm.fit(X_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
svm_tfidf_best_c = best_c_svm
svm_tfidf_model = final_model_svm

SVM TF-IDF fixed C=0.03:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed C: 0.03 (full search skipped)
Fold-by-fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.765252,0.563881,0.393417,0.314563,0.675743,0.277484,0.773714,1
1,0.768604,0.567256,0.387647,0.307628,0.684324,0.270414,0.773841,2
2,0.763538,0.561242,0.374723,0.294358,0.687595,0.257537,0.772578,3
3,0.764380,0.555682,0.387018,0.308636,0.671053,0.271922,0.772922,4
4,0.763328,0.562541,0.390511,0.311010,0.680376,0.273844,0.775002,5


Average fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.76502,0.56212,0.386663,0.307239,0.679818,0.27024,0.773611


#### 5.4.2 TF-IDF + Lemmatization

In [68]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

if USE_FIXED_BEST_PARAMS and SVM_TFIDF_LEMMA_FIXED_C is not None:
    best_c_svm_lemma = SVM_TFIDF_LEMMA_FIXED_C
    fold_rows = []
    for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"SVM TF-IDF+Lemma fixed C={best_c_svm_lemma:g}"):
        artifacts = load_cv_fold(fold_idx)
        X_train = artifacts["X_tfidf_lemma_train"]
        X_val = artifacts["X_tfidf_lemma_val"]
        y_train = artifacts["y_train"]
        y_val = artifacts["y_val"]
        svm = LinearSVC(C=best_c_svm_lemma, tol=SVM_TOL, max_iter=SVM_MAX_ITER, dual="auto", random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)
        model = CalibratedClassifierCV(svm, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_svm_lemma = pd.DataFrame(fold_rows)
    fixed_cv_summary_svm_lemma = fixed_cv_fold_details_svm_lemma[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed C: {best_c_svm_lemma} (full search skipped)")
    print("Fold-by-fold metrics for fixed C:")
    display(fixed_cv_fold_details_svm_lemma.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed C:")
    display(fixed_cv_summary_svm_lemma.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_svm_lemma = []
    cv_fold_details_by_c_svm_lemma = {}

    for c_value in tqdm(SVM_C_GRID, desc="SVM TF-IDF+Lemma C search"):
        fold_rows = []
        for fold_idx in tqdm(range(1, N_SPLITS + 1), desc=f"CV folds (C={c_value:g})", leave=False):
            artifacts = load_cv_fold(fold_idx)
            X_train = artifacts["X_tfidf_lemma_train"]
            X_val = artifacts["X_tfidf_lemma_val"]
            y_train = artifacts["y_train"]
            y_val = artifacts["y_val"]
            svm = LinearSVC(C=c_value, tol=SVM_TOL, max_iter=SVM_MAX_ITER, dual="auto", random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)
            model = CalibratedClassifierCV(svm, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
            model.fit(X_train, y_train)
            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)
        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_c_svm_lemma[c_value] = fold_df
        cv_summary_rows_svm_lemma.append({
            "C": c_value,
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_average_precision": fold_df["average_precision"].mean(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_f2": fold_df["f2"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        })

    cv_results_svm_lemma = pd.DataFrame(cv_summary_rows_svm_lemma).sort_values("mean_auc", ascending=False).reset_index(drop=True)
    print("CV summary across C values (ranked by mean AUC):")
    display(cv_results_svm_lemma.rename(columns={"mean_average_precision": "mean_pr-auc"}))
    best_c_svm_lemma = float(cv_results_svm_lemma.loc[0, "C"])
    print(f"\nBest C by CV mean AUC: {best_c_svm_lemma}")
    print("Fold-by-fold metrics for best C:")
    display(cv_fold_details_by_c_svm_lemma[best_c_svm_lemma].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_svm_lemma = cv_fold_details_by_c_svm_lemma[best_c_svm_lemma][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best C:")
    display(best_cv_summary_svm_lemma.rename(columns={"average_precision": "pr-auc"}))

artifacts = load_full_train_test()
X_train_full = artifacts["X_tfidf_lemma_train_full"]
y_train_full = artifacts["y_train_full"]
final_svm_lemma = LinearSVC(C=best_c_svm_lemma, tol=SVM_TOL, max_iter=SVM_MAX_ITER, dual="auto", random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)
final_model_svm_lemma = CalibratedClassifierCV(final_svm_lemma, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
final_model_svm_lemma.fit(X_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
svm_tfidf_lemma_best_c = best_c_svm_lemma
svm_tfidf_lemma_model = final_model_svm_lemma

SVM TF-IDF+Lemma fixed C=0.03:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed C: 0.03 (full search skipped)
Fold-by-fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.764174,0.562747,0.397248,0.317862,0.680509,0.280493,0.774896,1
1,0.767997,0.567130,0.389138,0.308792,0.687108,0.271430,0.774411,2
2,0.763012,0.561260,0.375459,0.294763,0.690531,0.257822,0.773041,3
3,0.764384,0.556553,0.388759,0.310062,0.673781,0.273193,0.773527,4
4,0.762714,0.562738,0.391294,0.311876,0.679813,0.274706,0.775034,5


Average fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.764456,0.562086,0.38838,0.308671,0.682348,0.271529,0.774182


#### 5.4.3 BERT Embeddings

In [69]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

artifacts = load_full_train_test()
X_bert_train_full = artifacts["X_bert_train_full"]
y_train_full = artifacts["y_train_full"]

if USE_FIXED_BEST_PARAMS and SVM_BERT_FIXED_C is not None:
    best_c_svm_bert = SVM_BERT_FIXED_C
    fold_rows = []
    for fold_idx, (train_idx, val_idx) in tqdm(
        enumerate(cv_splits, 1),
        total=len(cv_splits),
        desc=f"SVM BERT fixed C={best_c_svm_bert:g}",
    ):
        X_train = X_bert_train_full[train_idx]
        X_val = X_bert_train_full[val_idx]
        y_train = y_train_full[train_idx]
        y_val = y_train_full[val_idx]
        svm = make_pipeline(
            StandardScaler(),
            LinearSVC(
                C=best_c_svm_bert,
                tol=SVM_TOL,
                max_iter=SVM_MAX_ITER,
                dual="auto",
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            ),
        )
        model = CalibratedClassifierCV(svm, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_svm_bert = pd.DataFrame(fold_rows)
    fixed_cv_summary_svm_bert = fixed_cv_fold_details_svm_bert[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed C: {best_c_svm_bert} (full search skipped)")
    print("Fold-by-fold metrics for fixed C:")
    display(fixed_cv_fold_details_svm_bert.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed C:")
    display(fixed_cv_summary_svm_bert.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_svm_bert = []
    cv_fold_details_by_c_svm_bert = {}

    for c_value in tqdm(SVM_C_GRID, desc="SVM BERT C search"):
        fold_rows = []
        for fold_idx, (train_idx, val_idx) in tqdm(
            enumerate(cv_splits, 1),
            total=len(cv_splits),
            desc=f"CV folds (C={c_value:g})",
            leave=False,
        ):
            X_train = X_bert_train_full[train_idx]
            X_val = X_bert_train_full[val_idx]
            y_train = y_train_full[train_idx]
            y_val = y_train_full[val_idx]
            svm = make_pipeline(
                StandardScaler(),
                LinearSVC(
                    C=c_value,
                    tol=SVM_TOL,
                    max_iter=SVM_MAX_ITER,
                    dual="auto",
                    random_state=RANDOM_STATE,
                    class_weight=CLASS_WEIGHT,
                ),
            )
            model = CalibratedClassifierCV(svm, cv=SVM_CALIBRATION_CV, method=SVM_CALIBRATION_METHOD)
            model.fit(X_train, y_train)
            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)

        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_c_svm_bert[c_value] = fold_df
        cv_summary_rows_svm_bert.append({
            "C": c_value,
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_average_precision": fold_df["average_precision"].mean(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_f2": fold_df["f2"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        })

    cv_results_svm_bert = pd.DataFrame(cv_summary_rows_svm_bert).sort_values("mean_auc", ascending=False).reset_index(drop=True)
    print("CV summary across C values (ranked by mean AUC):")
    display(cv_results_svm_bert.rename(columns={"mean_average_precision": "mean_pr-auc"}))
    best_c_svm_bert = float(cv_results_svm_bert.loc[0, "C"])
    print(f"\nBest C by CV mean AUC: {best_c_svm_bert}")
    print("Fold-by-fold metrics for best C:")
    display(cv_fold_details_by_c_svm_bert[best_c_svm_bert].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_svm_bert = cv_fold_details_by_c_svm_bert[best_c_svm_bert][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best C:")
    display(best_cv_summary_svm_bert.rename(columns={"average_precision": "pr-auc"}))

final_svm_bert = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=best_c_svm_bert,
        tol=SVM_TOL,
        max_iter=SVM_MAX_ITER,
        dual="auto",
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
    ),
)
final_model_svm_bert = CalibratedClassifierCV(
    final_svm_bert,
    cv=SVM_CALIBRATION_CV,
    method=SVM_CALIBRATION_METHOD,
)
final_model_svm_bert.fit(X_bert_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
svm_bert_best_c = best_c_svm_bert
svm_bert_model = final_model_svm_bert

SVM BERT fixed C=3:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed C: 3.0 (full search skipped)
Fold-by-fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.723130,0.480895,0.255925,0.189133,0.622051,0.161103,0.751483,1
1,0.731772,0.495829,0.297334,0.227116,0.613424,0.196223,0.755595,2
2,0.729753,0.477369,0.279848,0.212077,0.598733,0.182597,0.761267,3
3,0.721061,0.485936,0.302025,0.232714,0.599729,0.201835,0.747276,4
4,0.721110,0.479013,0.282563,0.214875,0.594888,0.185286,0.748946,5


Average fold metrics for fixed C:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.725365,0.483808,0.283539,0.215183,0.605765,0.185409,0.752914


### 5.5 Decision Tree

#### 5.5.1 TF-IDF

In [70]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterSampler

DT_INT_PARAMS = ["max_depth", "min_samples_split", "min_samples_leaf"]
DT_FLOAT_PARAMS = ["ccp_alpha"]

def normalize_dt_params(params_like):
    params = dict(params_like)
    for name in DT_INT_PARAMS:
        value = params.get(name)
        params[name] = None if value is None or pd.isna(value) else int(value)
    for name in DT_FLOAT_PARAMS:
        value = params.get(name)
        params[name] = float(value)
    return params

if USE_FIXED_BEST_PARAMS and DT_TFIDF_FIXED_PARAMS is not None:
    best_params_dt = normalize_dt_params(DT_TFIDF_FIXED_PARAMS)
    fold_rows = []
    for fold_idx in tqdm(range(1, N_SPLITS + 1), desc="DT TF-IDF fixed params"):
        artifacts = load_cv_fold(fold_idx)
        X_train = artifacts["X_tfidf_train"]
        X_val = artifacts["X_tfidf_val"]
        y_train = artifacts["y_train"]
        y_val = artifacts["y_val"]
        model = DecisionTreeClassifier(
            max_depth=best_params_dt["max_depth"],
            min_samples_split=best_params_dt["min_samples_split"],
            min_samples_leaf=best_params_dt["min_samples_leaf"],
            ccp_alpha=best_params_dt["ccp_alpha"],
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_dt = pd.DataFrame(fold_rows)
    fixed_cv_summary_dt = fixed_cv_fold_details_dt[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed params: {best_params_dt} (full search skipped)")
    print("Fold-by-fold metrics for fixed params:")
    display(fixed_cv_fold_details_dt.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed params:")
    display(fixed_cv_summary_dt.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_dt = []
    cv_fold_details_by_params_dt = {}

    sampled_param_grid = list(
        ParameterSampler(
            DT_PARAM_GRID,
            n_iter=min(DT_RANDOM_SEARCH_ITER, np.prod([len(v) for v in DT_PARAM_GRID.values()])),
            random_state=RANDOM_STATE,
        )
    )

    print(f"Evaluating {len(sampled_param_grid)} sampled parameter settings instead of the full grid.")

    for params in tqdm(sampled_param_grid, desc="DT TF-IDF random search"):
        fold_rows = []
        for fold_idx in tqdm(
            range(1, N_SPLITS + 1),
            desc=f"CV folds {params}",
            leave=False,
        ):
            artifacts = load_cv_fold(fold_idx)
            X_train = artifacts["X_tfidf_train"]
            X_val = artifacts["X_tfidf_val"]
            y_train = artifacts["y_train"]
            y_val = artifacts["y_val"]
            model = DecisionTreeClassifier(
                max_depth=params["max_depth"],
                min_samples_split=params["min_samples_split"],
                min_samples_leaf=params["min_samples_leaf"],
                ccp_alpha=params["ccp_alpha"],
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            )
            model.fit(X_train, y_train)
            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)
        params_key = tuple(params[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"])
        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_params_dt[params_key] = fold_df
        cv_summary_rows_dt.append({
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "min_samples_leaf": params["min_samples_leaf"],
            "ccp_alpha": params["ccp_alpha"],
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_average_precision": fold_df["average_precision"].mean(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_f2": fold_df["f2"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        })

    cv_results_dt = pd.DataFrame(cv_summary_rows_dt).sort_values("mean_auc", ascending=False).reset_index(drop=True)
    print("CV summary across sampled hyperparameters (ranked by mean AUC):")
    display(cv_results_dt.rename(columns={"mean_average_precision": "mean_pr-auc"}))
    best_params_dt = normalize_dt_params(cv_results_dt.loc[0, ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"]])
    print(f"\nBest params by CV mean AUC: {best_params_dt}")
    print("Fold-by-fold metrics for best params:")
    display(cv_fold_details_by_params_dt[tuple(best_params_dt[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"] )].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_dt = cv_fold_details_by_params_dt[tuple(best_params_dt[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"] )][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best params:")
    display(best_cv_summary_dt.rename(columns={"average_precision": "pr-auc"}))

artifacts = load_full_train_test()
X_train_full = artifacts["X_tfidf_train_full"]
y_train_full = artifacts["y_train_full"]
final_model_dt = DecisionTreeClassifier(
    max_depth=best_params_dt["max_depth"],
    min_samples_split=best_params_dt["min_samples_split"],
    min_samples_leaf=best_params_dt["min_samples_leaf"],
    ccp_alpha=best_params_dt["ccp_alpha"],
    random_state=RANDOM_STATE,
    class_weight=CLASS_WEIGHT,
 )
final_model_dt.fit(X_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
dt_tfidf_best_params = dict(best_params_dt)
dt_tfidf_model = final_model_dt

DT TF-IDF fixed params:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed params: {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001} (full search skipped)
Fold-by-fold metrics for fixed params:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.690375,0.465412,0.484747,0.552005,0.402925,0.608270,0.658033,1
1,0.688025,0.461160,0.481189,0.543435,0.404054,0.594724,0.660509,2
2,0.686809,0.461897,0.476791,0.531462,0.407010,0.575451,0.665816,3
3,0.688047,0.461191,0.478484,0.541917,0.400376,0.594455,0.658388,4
4,0.687522,0.464269,0.477563,0.540920,0.399563,0.593403,0.658259,5


Average fold metrics for fixed params:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.688156,0.462786,0.479755,0.541948,0.402785,0.593261,0.660201


#### 5.5.2 TF-IDF + Lemmatization

In [71]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterSampler

if USE_FIXED_BEST_PARAMS and DT_TFIDF_LEMMA_FIXED_PARAMS is not None:
    best_params_dt_lemma = normalize_dt_params(DT_TFIDF_LEMMA_FIXED_PARAMS)
    fold_rows = []
    for fold_idx in tqdm(range(1, N_SPLITS + 1), desc="DT TF-IDF+Lemma fixed params"):
        artifacts = load_cv_fold(fold_idx)
        X_train = artifacts["X_tfidf_lemma_train"]
        X_val = artifacts["X_tfidf_lemma_val"]
        y_train = artifacts["y_train"]
        y_val = artifacts["y_val"]
        model = DecisionTreeClassifier(
            max_depth=best_params_dt_lemma["max_depth"],
            min_samples_split=best_params_dt_lemma["min_samples_split"],
            min_samples_leaf=best_params_dt_lemma["min_samples_leaf"],
            ccp_alpha=best_params_dt_lemma["ccp_alpha"],
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_dt_lemma = pd.DataFrame(fold_rows)
    fixed_cv_summary_dt_lemma = fixed_cv_fold_details_dt_lemma[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed params: {best_params_dt_lemma} (full search skipped)")
    print("Fold-by-fold metrics for fixed params:")
    display(fixed_cv_fold_details_dt_lemma.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed params:")
    display(fixed_cv_summary_dt_lemma.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_dt_lemma = []
    cv_fold_details_by_params_dt_lemma = {}

    sampled_param_grid = list(
        ParameterSampler(
            DT_PARAM_GRID,
            n_iter=min(DT_RANDOM_SEARCH_ITER, np.prod([len(v) for v in DT_PARAM_GRID.values()])),
            random_state=RANDOM_STATE,
        )
    )

    print(f"Evaluating {len(sampled_param_grid)} sampled parameter settings instead of the full grid.")

    for params in tqdm(sampled_param_grid, desc="DT TF-IDF+Lemma random search"):
        fold_rows = []
        for fold_idx in tqdm(
            range(1, N_SPLITS + 1),
            desc=f"CV folds {params}",
            leave=False,
        ):
            artifacts = load_cv_fold(fold_idx)
            X_train = artifacts["X_tfidf_lemma_train"]
            X_val = artifacts["X_tfidf_lemma_val"]
            y_train = artifacts["y_train"]
            y_val = artifacts["y_val"]
            model = DecisionTreeClassifier(
                max_depth=params["max_depth"],
                min_samples_split=params["min_samples_split"],
                min_samples_leaf=params["min_samples_leaf"],
                ccp_alpha=params["ccp_alpha"],
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            )
            model.fit(X_train, y_train)
            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)
        params_key = tuple(params[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"])
        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_params_dt_lemma[params_key] = fold_df
        cv_summary_rows_dt_lemma.append({
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "min_samples_leaf": params["min_samples_leaf"],
            "ccp_alpha": params["ccp_alpha"],
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_average_precision": fold_df["average_precision"].mean(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_f2": fold_df["f2"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        })

    cv_results_dt_lemma = pd.DataFrame(cv_summary_rows_dt_lemma).sort_values("mean_auc", ascending=False).reset_index(drop=True)
    print("CV summary across sampled hyperparameters (ranked by mean AUC):")
    display(cv_results_dt_lemma.rename(columns={"mean_average_precision": "mean_pr-auc"}))
    best_params_dt_lemma = normalize_dt_params(cv_results_dt_lemma.loc[0, ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"]])
    print(f"\nBest params by CV mean AUC: {best_params_dt_lemma}")
    print("Fold-by-fold metrics for best params:")
    display(cv_fold_details_by_params_dt_lemma[tuple(best_params_dt_lemma[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"] )].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_dt_lemma = cv_fold_details_by_params_dt_lemma[tuple(best_params_dt_lemma[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"] )][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best params:")
    display(best_cv_summary_dt_lemma.rename(columns={"average_precision": "pr-auc"}))

artifacts = load_full_train_test()
X_train_full = artifacts["X_tfidf_lemma_train_full"]
y_train_full = artifacts["y_train_full"]
final_model_dt_lemma = DecisionTreeClassifier(
    max_depth=best_params_dt_lemma["max_depth"],
    min_samples_split=best_params_dt_lemma["min_samples_split"],
    min_samples_leaf=best_params_dt_lemma["min_samples_leaf"],
    ccp_alpha=best_params_dt_lemma["ccp_alpha"],
    random_state=RANDOM_STATE,
    class_weight=CLASS_WEIGHT,
 )
final_model_dt_lemma.fit(X_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
dt_tfidf_lemma_best_params = dict(best_params_dt_lemma)
dt_tfidf_lemma_model = final_model_dt_lemma

DT TF-IDF+Lemma fixed params:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed params: {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001} (full search skipped)
Fold-by-fold metrics for fixed params:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.693332,0.469128,0.482066,0.539222,0.409690,0.585502,0.667280,1
1,0.692990,0.466282,0.480244,0.530273,0.414989,0.569849,0.673474,2
2,0.691681,0.467852,0.479028,0.547412,0.396480,0.604988,0.651799,3
3,0.690901,0.466109,0.480897,0.550947,0.396810,0.610204,0.652712,4
4,0.689197,0.463241,0.473723,0.515340,0.417526,0.547400,0.679860,5


Average fold metrics for fixed params:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.69162,0.466522,0.479192,0.536639,0.407099,0.583589,0.665025


#### 5.5.3 BERT Embeddings

In [72]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import ParameterSampler

artifacts = load_full_train_test()
X_bert_train_full = artifacts["X_bert_train_full"]
y_train_full = artifacts["y_train_full"]

if USE_FIXED_BEST_PARAMS and DT_BERT_FIXED_PARAMS is not None:
    best_params_dt_bert = normalize_dt_params(DT_BERT_FIXED_PARAMS)
    fold_rows = []
    for fold_idx, (train_idx, val_idx) in tqdm(
        enumerate(cv_splits, 1),
        total=len(cv_splits),
        desc="DT BERT fixed params",
    ):
        X_train = X_bert_train_full[train_idx]
        X_val = X_bert_train_full[val_idx]
        y_train = y_train_full[train_idx]
        y_val = y_train_full[val_idx]
        model = DecisionTreeClassifier(
            max_depth=best_params_dt_bert["max_depth"],
            min_samples_split=best_params_dt_bert["min_samples_split"],
            min_samples_leaf=best_params_dt_bert["min_samples_leaf"],
            ccp_alpha=best_params_dt_bert["ccp_alpha"],
            random_state=RANDOM_STATE,
            class_weight=CLASS_WEIGHT,
        )
        model.fit(X_train, y_train)
        y_val_prob = model.predict_proba(X_val)[:, 1]
        metrics = {
            "auc": roc_auc_score(y_val, y_val_prob),
            "average_precision": average_precision_score(y_val, y_val_prob),
            "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
            "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
            "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
            "fold": fold_idx,
        }
        fold_rows.append(metrics)

    fixed_cv_fold_details_dt_bert = pd.DataFrame(fold_rows)
    fixed_cv_summary_dt_bert = fixed_cv_fold_details_dt_bert[["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print(f"Using fixed params: {best_params_dt_bert} (full search skipped)")
    print("Fold-by-fold metrics for fixed params:")
    display(fixed_cv_fold_details_dt_bert.rename(columns={"average_precision": "pr-auc"}))
    print("Average fold metrics for fixed params:")
    display(fixed_cv_summary_dt_bert.rename(columns={"average_precision": "pr-auc"}))
else:
    cv_summary_rows_dt_bert = []
    cv_fold_details_by_params_dt_bert = {}

    sampled_param_grid = list(
        ParameterSampler(
            DT_PARAM_GRID,
            n_iter=min(DT_RANDOM_SEARCH_ITER, np.prod([len(v) for v in DT_PARAM_GRID.values()])),
            random_state=RANDOM_STATE,
        )
    )

    print(f"Evaluating {len(sampled_param_grid)} sampled parameter settings instead of the full grid.")

    for params in tqdm(sampled_param_grid, desc="DT BERT random search"):
        fold_rows = []
        for fold_idx, (train_idx, val_idx) in tqdm(
            enumerate(cv_splits, 1),
            total=len(cv_splits),
            desc=f"CV folds {params}",
            leave=False,
        ):
            X_train = X_bert_train_full[train_idx]
            X_val = X_bert_train_full[val_idx]
            y_train = y_train_full[train_idx]
            y_val = y_train_full[val_idx]
            model = DecisionTreeClassifier(
                max_depth=params["max_depth"],
                min_samples_split=params["min_samples_split"],
                min_samples_leaf=params["min_samples_leaf"],
                ccp_alpha=params["ccp_alpha"],
                random_state=RANDOM_STATE,
                class_weight=CLASS_WEIGHT,
            )
            model.fit(X_train, y_train)
            y_val_prob = model.predict_proba(X_val)[:, 1]
            metrics = {
                "auc": roc_auc_score(y_val, y_val_prob),
                "average_precision": average_precision_score(y_val, y_val_prob),
                "f1": f1_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "f2": fbeta_score(y_val, y_val_prob >= 0.5, beta=2, zero_division=0),
                "precision": precision_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "recall": recall_score(y_val, y_val_prob >= 0.5, zero_division=0),
                "accuracy": accuracy_score(y_val, y_val_prob >= 0.5),
                "fold": fold_idx,
            }
            fold_rows.append(metrics)
        params_key = tuple(params[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"])
        fold_df = pd.DataFrame(fold_rows)
        cv_fold_details_by_params_dt_bert[params_key] = fold_df
        cv_summary_rows_dt_bert.append({
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "min_samples_leaf": params["min_samples_leaf"],
            "ccp_alpha": params["ccp_alpha"],
            "mean_auc": fold_df["auc"].mean(),
            "std_auc": fold_df["auc"].std(),
            "mean_average_precision": fold_df["average_precision"].mean(),
            "mean_f1": fold_df["f1"].mean(),
            "mean_f2": fold_df["f2"].mean(),
            "mean_precision": fold_df["precision"].mean(),
            "mean_recall": fold_df["recall"].mean(),
            "mean_accuracy": fold_df["accuracy"].mean(),
        })

    cv_results_dt_bert = pd.DataFrame(cv_summary_rows_dt_bert).sort_values("mean_auc", ascending=False).reset_index(drop=True)
    print("CV summary across sampled hyperparameters (ranked by mean AUC):")
    display(cv_results_dt_bert.rename(columns={"mean_average_precision": "mean_pr-auc"}))
    best_params_dt_bert = normalize_dt_params(cv_results_dt_bert.loc[0, ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"]])
    print(f"\nBest params by CV mean AUC: {best_params_dt_bert}")
    print("Fold-by-fold metrics for best params:")
    display(cv_fold_details_by_params_dt_bert[tuple(best_params_dt_bert[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"] )].rename(columns={"average_precision": "pr-auc"}))
    best_cv_summary_dt_bert = cv_fold_details_by_params_dt_bert[tuple(best_params_dt_bert[name] for name in ["max_depth", "min_samples_split", "min_samples_leaf", "ccp_alpha"] )][["auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].mean().to_frame().T
    print("Average fold metrics for best params:")
    display(best_cv_summary_dt_bert.rename(columns={"average_precision": "pr-auc"}))

final_model_dt_bert = DecisionTreeClassifier(
    max_depth=best_params_dt_bert["max_depth"],
    min_samples_split=best_params_dt_bert["min_samples_split"],
    min_samples_leaf=best_params_dt_bert["min_samples_leaf"],
    ccp_alpha=best_params_dt_bert["ccp_alpha"],
    random_state=RANDOM_STATE,
    class_weight=CLASS_WEIGHT,
 )
final_model_dt_bert.fit(X_bert_train_full, y_train_full)

# Keep model and selected hyperparameter available for model evaluation (section 6)
dt_bert_best_params = dict(best_params_dt_bert)
dt_bert_model = final_model_dt_bert

DT BERT fixed params:   0%|          | 0/5 [00:00<?, ?it/s]

Using fixed params: {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'ccp_alpha': 0.0001} (full search skipped)
Fold-by-fold metrics for fixed params:


,auc,pr-auc,f1,f2,precision,recall,accuracy,fold
0,0.654017,0.380034,0.460652,0.564084,0.352827,0.663385,0.587894,1
1,0.664565,0.388064,0.471320,0.591533,0.352071,0.712724,0.578639,2
2,0.655104,0.363175,0.452548,0.558849,0.343614,0.662611,0.592754,3
3,0.651672,0.384114,0.468349,0.577936,0.355879,0.684752,0.578841,4
4,0.653694,0.379131,0.466337,0.577183,0.353264,0.685869,0.581143,5


Average fold metrics for fixed params:


,auc,pr-auc,f1,f2,precision,recall,accuracy
0,0.65581,0.378904,0.463841,0.573917,0.351531,0.681868,0.583854


## 6. Final Test Set Evaluation

This section reports held-out test performance after hyperparameter tuning and final refitting are complete. Results are compared across the three feature representations: TF-IDF, TF-IDF + lemmatization, and BERT embeddings.

### 6.1 Logistic Regression

In [73]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

artifacts = load_full_train_test()

# TF-IDF
X_test_tfidf = artifacts["X_tfidf_test"]
y_test = artifacts["y_test"]
y_test_prob_tfidf = logreg_tfidf_model.predict_proba(X_test_tfidf)[:, 1]
test_metrics_tfidf = compute_metrics(y_test, y_test_prob_tfidf)

# TF-IDF + Lemma
X_test_lemma = artifacts["X_tfidf_lemma_test"]
y_test_prob_lemma = logreg_tfidf_lemma_model.predict_proba(X_test_lemma)[:, 1]
test_metrics_lemma = compute_metrics(y_test, y_test_prob_lemma)

# BERT
X_test_bert = artifacts["X_bert_test"]
y_test_prob_bert = logreg_bert_model.predict_proba(X_test_bert)[:, 1]
test_metrics_bert = compute_metrics(y_test, y_test_prob_bert)

# Compare test results
test_metrics_df = pd.DataFrame([
    {"feature": "TF-IDF", **test_metrics_tfidf},
    {"feature": "TF-IDF+Lemma", **test_metrics_lemma},
    {"feature": "BERT", **test_metrics_bert}
])
print("\nFinal held-out test metrics (Logistic Regression):")
display(test_metrics_df[["feature", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].rename(columns={"average_precision": "PR-AUC"}))


Final held-out test metrics (Logistic Regression):


,feature,auc,PR-AUC,f1,f2,precision,recall,accuracy
0,TF-IDF,0.769518,0.560090,0.547799,0.604941,0.473288,0.650154,0.722874
1,TF-IDF+Lemma,0.768899,0.560014,0.544398,0.600414,0.471139,0.644634,0.721431
2,BERT,0.726797,0.472879,0.505393,0.591910,0.406392,0.668165,0.662352


### 6.2 Support Vector Machine

In [74]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

artifacts = load_full_train_test()

# TF-IDF
X_test_tfidf = artifacts["X_tfidf_test"]
y_test = artifacts["y_test"]
y_test_prob_svm_tfidf = svm_tfidf_model.predict_proba(X_test_tfidf)[:, 1]
test_metrics_svm_tfidf = compute_metrics(y_test, y_test_prob_svm_tfidf)

# TF-IDF + Lemma
X_test_lemma = artifacts["X_tfidf_lemma_test"]
y_test_prob_svm_lemma = svm_tfidf_lemma_model.predict_proba(X_test_lemma)[:, 1]
test_metrics_svm_lemma = compute_metrics(y_test, y_test_prob_svm_lemma)

# BERT
X_test_bert = artifacts["X_bert_test"]
y_test_prob_svm_bert = svm_bert_model.predict_proba(X_test_bert)[:, 1]
test_metrics_svm_bert = compute_metrics(y_test, y_test_prob_svm_bert)

# Combine results
test_metrics_svm_df = pd.DataFrame([
    {"feature": "TF-IDF", **test_metrics_svm_tfidf},
    {"feature": "TF-IDF+Lemma", **test_metrics_svm_lemma},
    {"feature": "BERT", **test_metrics_svm_bert}
])
print("\nFinal held-out test metrics (SVM):")
display(test_metrics_svm_df[["feature", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].rename(columns={"average_precision": "PR-AUC"}))


Final held-out test metrics (SVM):


,feature,auc,PR-AUC,f1,f2,precision,recall,accuracy
0,TF-IDF,0.769418,0.557287,0.395911,0.317872,0.670099,0.280953,0.778647
1,TF-IDF+Lemma,0.768828,0.557439,0.392983,0.315024,0.668850,0.278228,0.778089
2,BERT,0.727075,0.472891,0.287527,0.220461,0.583234,0.190793,0.755881


### 6.3 Decision Tree

In [75]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

artifacts = load_full_train_test()

# TF-IDF
X_test_tfidf = artifacts["X_tfidf_test"]
y_test = artifacts["y_test"]
y_test_prob_dt_tfidf = dt_tfidf_model.predict_proba(X_test_tfidf)[:, 1]
test_metrics_dt_tfidf = compute_metrics(y_test, y_test_prob_dt_tfidf)

# TF-IDF + Lemmatization
X_test_lemma = artifacts["X_tfidf_lemma_test"]
y_test_prob_dt_tfidf_lemma = dt_tfidf_lemma_model.predict_proba(X_test_lemma)[:, 1]
test_metrics_dt_lemma = compute_metrics(y_test, y_test_prob_dt_tfidf_lemma)

# BERT
X_test_bert = artifacts["X_bert_test"]
y_test_prob_dt_bert = dt_bert_model.predict_proba(X_test_bert)[:, 1]
test_metrics_dt_bert = compute_metrics(y_test, y_test_prob_dt_bert)

# Combine results
test_metrics_dt_df = pd.DataFrame([
    {"feature": "TF-IDF", **test_metrics_dt_tfidf},
    {"feature": "TF-IDF+Lemma", **test_metrics_dt_lemma},
    {"feature": "BERT", **test_metrics_dt_bert}
])
print("\nFinal held-out test metrics (Decision Tree):")
display(test_metrics_dt_df[["feature", "auc", "average_precision", "f1", "f2", "precision", "recall", "accuracy"]].rename(columns={"average_precision": "PR-AUC"}))


Final held-out test metrics (Decision Tree):


,feature,auc,PR-AUC,f1,f2,precision,recall,accuracy
0,TF-IDF,0.692560,0.462521,0.478229,0.546671,0.395668,0.604331,0.659538
1,TF-IDF+Lemma,0.695337,0.465645,0.474830,0.530542,0.404104,0.575563,0.671295
2,BERT,0.664972,0.373711,0.457932,0.547098,0.360113,0.628711,0.615715
